# 01 Explore Baseline Retrieval

Use this notebook to inspect Stage 1 Semantic Scholar retrieval outputs before reranking.

## What This Notebook Does
This notebook focuses on Stage 1 only: baseline retrieval from Semantic Scholar.

High-level flow:
1. Set up project import paths inside the notebook runtime.
2. Load query specifications from JSON.
3. Retrieve top-K candidate papers for one selected query.
4. Inspect and optionally save the baseline candidate list.
        

### Step 1: Notebook Runtime Setup
This cell resolves the project root and adds it to `sys.path` so `from src...` imports work reliably inside Jupyter.
        

In [10]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

Project root: /Users/juno/paper-reranking-demo


### Step 2: Import Pipeline Components
This cell imports config values, query loading utilities, and retrieval classes used in the next steps.
        

In [11]:
import pandas as pd

from src import config
from src.schema.query_instantiator import load_query_specs
from src.retrieval.semantic_scholar_client import SemanticScholarClient
from src.retrieval.candidate_retriever import CandidateRetriever

### Step 3: Load and Preview Query Specs
This cell reads `data/queries/example_queries.json`, converts each query spec to a row, and displays the schema-instantiated inputs.
        

In [12]:
from dataclasses import asdict
query_specs = load_query_specs(config.DEFAULT_QUERY_FILE)
queries_df = pd.DataFrame([asdict(q) for q in query_specs])
queries_df

,raw_query,topic_text,method_text,relationship_text,recency_preference
0,Graph neural networks for molecular property p...,molecular property prediction using graph neur...,message passing neural networks and graph repr...,relationship between molecular graph structure...,recent
1,RAG methods for scientific question answering,retrieval-augmented generation for scientific QA,dense retrieval plus generation with citations,how retrieval quality impacts factuality and a...,recent
2,Causal inference in education policy,education policy evaluation and causal inference,difference-in-differences and synthetic contro...,relationship between policy interventions and ...,balanced


### Step 4: Run Baseline Retrieval for One Query
This cell selects one query, calls Semantic Scholar retrieval for `TOP_K`, and stores the raw candidate `Paper` objects.
        

In [13]:
TOP_K = 10
query_idx = 0

query_spec = query_specs[query_idx]
print(f"Running baseline retrieval for query: {query_spec.raw_query}")

client = SemanticScholarClient(api_key=config.SEMANTIC_SCHOLAR_API_KEY)
retriever = CandidateRetriever(client=client)

candidates = retriever.retrieve_candidates(raw_query=query_spec.raw_query, top_k=TOP_K)
print(f"Retrieved {len(candidates)} candidates")

Running baseline retrieval for query: Graph neural networks for molecular property prediction
Retrieved 10 candidates


### Step 5: Build an Inspectable Baseline Table
This cell converts candidate objects into a DataFrame with rank/title/year/venue so you can quickly inspect retrieval quality.
        

In [14]:
baseline_rows = [
    {
        "baseline_rank": idx + 1,
        "paper_id": p.paper_id,
        "title": p.title,
        "year": p.year,
        "venue": p.venue,
    }
    for idx, p in enumerate(candidates)
]

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.head(20)

,baseline_rank,paper_id,title,year,venue
0,1,8495965cf4ff5eb8010c7f126690106c135f23ab,Kolmogorov–Arnold graph neural networks for mo...,2025,Nature Machine Intelligence
1,2,ee75d0675c7adedded789e38500cc0b32b9fb4ae,Chemistry-intuitive explanation of graph neura...,2023,Nature Communications
2,3,c639624193847693bb2fd4b1319ca3229093c412,Mfgnn: Multi‐Scale Feature‐Attentive Graph Neu...,2025,Journal of Computational Chemistry
3,4,7675fc907efe86a45f3259784d644897dc95af1b,Quantitative evaluation of explainable graph n...,2021,Patterns
4,5,049c6ba3c6776e474cd2b60d081f47299f85091a,Physical Pooling Functions in Graph Neural Net...,2022,Computers and Chemical Engineering
5,6,b7fdef4f00bed1ebb70d8d86534bfd76a27ffb6c,Consistency-regularized graph neural networks ...,2025,Neural Networks
6,7,6039ef4676d535ef5b60f6315036a84b9d5ebe3b,Chain-aware graph neural networks for molecula...,2024,Bioinform.
7,8,be5d48ff7434525a965390f95ceb9902a46e76e0,Multi-View Graph Neural Networks for Molecular...,2020,
8,9,c70d99bc680615b38111f949ce1ff0cc171d3967,Cross-dependent graph neural networks for mole...,2022,Bioinform.
9,10,82a4356ab71189ae9837b7752ba31fa570a04514,Composite Graph Neural Networks for Molecular ...,2024,International Journal of Molecular Sciences


### Step 6: Save Baseline Results
This cell writes `data/outputs/baseline_candidates.csv` if any rows were retrieved.
        

In [15]:
if not baseline_df.empty:
    out_path = ROOT / "data" / "outputs" / "notebook_baseline_preview.csv"
    baseline_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
else:
    print("No candidates retrieved. Check API key and network connectivity.")

Saved: /Users/juno/paper-reranking-demo/data/outputs/notebook_baseline_preview.csv
